In [5]:
words = open('name.txt', 'r').read().splitlines()
words[:10]

['emma',
 'olivia',
 'ava',
 'isabella',
 'sophia',
 'charlotte',
 'mia',
 'amelia',
 'harper',
 'evelyn']

In [38]:
# FIRST APPROACH: STATISTICAL ------------------------->

In [7]:
import torch

In [8]:
N2 = torch.zeros((27, 27), dtype=torch.int32)
N3 = torch.zeros((27, 27, 27), dtype=torch.int32)

In [9]:
chars = sorted(list(set(''.join(words))))
stoi = {s : i+1 for i, s in enumerate(chars)}
stoi['.'] = 0
itos = {i : s for s, i in stoi.items()}

for w in words:
    chs = ["."] + list(w) + ["."]
    for ch1, ch2 in zip(chs, chs[1:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        N2[ix1, ix2] += 1

for w in words:
    chs = ["."] + list(w) + ["."]
    for ch1, ch2, ch3 in zip(chs, chs[1:], chs[2:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        ix3 = stoi[ch3]
        N3[ix1, ix2, ix3] += 1

In [10]:
# how to use N?
# N[1, 2, 3] would be the count of 'abc' in the training data set
# N[1, 2].sum() would be the count of trigrams starting with 'ab'
# N[1].sum() woule be the count of trigrams starting with 'a'
#
# we know the first character is .
# so we start at P2[0], and pick the first ix1
# then we use P[ix1] to pick the second ix2, by multinomial
# then normally, we already know the first two character and want to pick the third
# we use P3[ix1, ix2] to multinomially pick the third
P2 = (N2 + 1).float()
P2 /= P2.sum(1, keepdim=True)
P3 = (N3 + 1).float()
P3 /= P3.sum(2, keepdim=True)

In [43]:
t = torch.arange(27, dtype=torch.int32).reshape(3, 3, 3)
print(t)
p3 = (t + 1).float()
p3 /= p3.sum(2, keepdim=True)
print(p3)

s = torch.arange(9, dtype=torch.int32).reshape(3, 3)
print(s)
p2 = (s + 1).float()
p2 /= p2.sum(1, keepdim=True)
print(p2)

tensor([[[ 0,  1,  2],
         [ 3,  4,  5],
         [ 6,  7,  8]],

        [[ 9, 10, 11],
         [12, 13, 14],
         [15, 16, 17]],

        [[18, 19, 20],
         [21, 22, 23],
         [24, 25, 26]]], dtype=torch.int32)
tensor([[[0.1667, 0.3333, 0.5000],
         [0.2667, 0.3333, 0.4000],
         [0.2917, 0.3333, 0.3750]],

        [[0.3030, 0.3333, 0.3636],
         [0.3095, 0.3333, 0.3571],
         [0.3137, 0.3333, 0.3529]],

        [[0.3167, 0.3333, 0.3500],
         [0.3188, 0.3333, 0.3478],
         [0.3205, 0.3333, 0.3462]]])
tensor([[0, 1, 2],
        [3, 4, 5],
        [6, 7, 8]], dtype=torch.int32)
tensor([[0.1667, 0.3333, 0.5000],
        [0.2667, 0.3333, 0.4000],
        [0.2917, 0.3333, 0.3750]])


In [44]:
for i in range(5):
    out = []
    # pick first character
    p = P2[0]
    ix = torch.multinomial(p, num_samples=1, replacement=True).item()
    out.append(itos[ix])
    if ix == 0:
        continue
    
    # pick second character
    p = P2[ix]
    ix = torch.multinomial(p, num_samples=1, replacement=True).item()
    out.append(itos[ix])
    if ix == 0:
        continue

    while True:
        ix1 = stoi[out[-2]]
        ix2 = stoi[out[-1]]
        p = P3[ix1, ix2]
        ix3 = torch.multinomial(p, num_samples=1, replacement=True).item()
        out.append(itos[ix3])
        if ix3 == 0:
            break
    print("".join(out))

ada.
hamanakeilicamehbce.
slaholancarielluwa.
daviahi.
inder.


In [53]:
# loss function
log_likelihood = 0.0
n = 0
for w in words:
    chs = ["."] + list(w) + ["."]
    for ch1, ch2, ch3 in zip(chs, chs[1:], chs[2:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        ix3 = stoi[ch3]
        prob = P3[ix1, ix2, ix3]
        logprob = torch.log(prob)
        log_likelihood += logprob
        n += 1

print(f'{log_likelihood=}')
nll = -log_likelihood
print(f'{nll=}')
print(f'{nll/n}')

log_likelihood=tensor(-410414.9688)
nll=tensor(410414.9688)
2.092747449874878


In [36]:
# now xs, ys are the two input chars, zs is the label (char to predict)
xs, ys, zs = [], [], []
for w in words:
    chs = ["."] + list(w) + ["."]
    for ch1, ch2, ch3 in zip(chs, chs[1:], chs[2:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        ix3 = stoi[ch3]
        xs.append(ix1)
        ys.append(ix2)
        zs.append(ix3)

xs = torch.tensor(xs)
ys = torch.tensor(ys)
zs = torch.tensor(zs)
num = xs.nelement()
print('xs:', xs[:5])
print('ys:', ys[:5])
print('zs:', zs[:5])
print('number of examples: ', num)

xs: tensor([ 0,  5, 13, 13,  0])
ys: tensor([ 5, 13, 13,  1, 15])
zs: tensor([13, 13,  1,  0, 12])
number of examples:  196113


In [37]:
# initialize the network
# the input is now TWO characters. each is one-hot encoded into a
# length-27 vector, and we concatenate them into one length-54 input.
# the output layer still has 27 neurons (one per possible next char),
# so the weight matrix W has shape (54, 27).
#   (num, 54) @ (54, 27) -> (num, 27)
g = torch.Generator().manual_seed(2147483647)
W = torch.randn((54, 27), generator=g, requires_grad=True)

In [38]:
import torch.nn.functional as F

In [39]:
# gradient descent
for k in range(100):

    # forward pass
    xenc = F.one_hot(xs, num_classes=27).float()   # (num, 27) one-hot of 1st char
    yenc = F.one_hot(ys, num_classes=27).float()   # (num, 27) one-hot of 2nd char
    xyenc = torch.cat([xenc, yenc], dim=1)         # (num, 54) concatenated input
    logits = xyenc @ W   # (num, 27) interpreted as log-counts; predict log-counts
    counts = logits.exp()   # (num, 27) equivalent to N
    probs = counts / counts.sum(1, keepdim=True) # (num, 27) probabilities for next char
    # negative log-likelihood of the true next char (zs), plus L2 regularization
    loss = -probs[torch.arange(num), zs].log().mean() + 0.01 * (W**2).mean()
    print(loss.item())

    # backward pass
    W.grad = None  # set to zero the gradient
    loss.backward()

    # update
    W.data += -50 * W.grad

4.1959710121154785
3.3653783798217773
3.0495336055755615
2.878478765487671
2.7739572525024414
2.7012386322021484
2.6454994678497314
2.601283073425293
2.5652337074279785
2.535409450531006
2.51039719581604
2.4892261028289795
2.4711146354675293
2.455474853515625
2.4418272972106934
2.4298086166381836
2.4191300868988037
2.4095730781555176
2.400963068008423
2.3931641578674316
2.386065721511841
2.3795783519744873
2.373626708984375
2.368147611618042
2.3630881309509277
2.358403205871582
2.354053020477295
2.3500046730041504
2.3462281227111816
2.342698097229004
2.3393921852111816
2.3362905979156494
2.3333756923675537
2.330632209777832
2.3280460834503174
2.3256044387817383
2.3232967853546143
2.321112632751465
2.3190431594848633
2.317079782485962
2.3152151107788086
2.313441753387451
2.311753988265991
2.3101460933685303
2.308612585067749
2.3071482181549072
2.3057491779327393
2.3044111728668213
2.303130626678467
2.301903486251831
2.3007266521453857
2.29959774017334
2.298513412475586
2.297471284866333